# 04 Statistical Analysis

Keep this notebook focused on basic statistical checks: overall performance, simple segment comparisons, and easy-to-read numeric differences.

In [8]:
from pathlib import Path

import pandas as pd
from scipy import stats
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / 'data/processed/cleaned_dataset.csv'

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

stats_df = pd.read_csv(DATA_PATH)
stats_df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y,previously_contacted,duration_minutes,age_group,balance_band,campaign_band
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,0,0,4.35,55-64,medium,1
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,0,0,2.52,35-44,low,1
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,0,0,1.27,25-34,low,1
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,0,0,1.53,45-54,medium,1
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,0,0,3.30,25-34,low,1


In [11]:
overall_summary = pd.DataFrame(
    {
        'metric': ['Total contacts', 'Total subscriptions', 'Subscription rate'],
        'value': [
            len(stats_df),
            int(stats_df['y'].sum()),
            stats_df['y'].mean(),
        ],
    }
)

job_subscription = (
    stats_df.groupby('job', dropna=False)['y']
    .mean()
    .sort_values(ascending=False)
    .reset_index(name='subscription_rate')
)

month_subscription = (
    stats_df.groupby(['month'], dropna=False)['y']
    .mean()
    .reset_index(name='subscription_rate')
    .sort_values('month')
)

display(overall_summary)
display(job_subscription)
display(month_subscription)


,metric,value
0,Total contacts,45211.000000
1,Total subscriptions,5289.000000
2,Subscription rate,0.116985


,job,subscription_rate
0,student,0.286780
1,retired,0.227915
2,unemployed,0.155027
3,management,0.137556
4,admin.,0.122027
5,self-employed,0.118429
6,unknown,0.118056
7,technician,0.110570
8,services,0.088830
9,housemaid,0.087903


,month,subscription_rate
0,apr,0.196794
1,aug,0.110133
2,dec,0.467290
3,feb,0.166478
4,jan,0.101212
5,jul,0.090935
6,jun,0.102228
7,mar,0.519916
8,may,0.067195
9,nov,0.101511


In [12]:
subscribed = stats_df.loc[stats_df['y'] == 1]
not_subscribed = stats_df.loc[stats_df['y'] == 0]

age_p = stats.mannwhitneyu(subscribed['age'], not_subscribed['age'], alternative='two-sided').pvalue
balance_p = stats.mannwhitneyu(subscribed['balance'], not_subscribed['balance'], alternative='two-sided').pvalue
campaign_p = stats.mannwhitneyu(subscribed['campaign'], not_subscribed['campaign'], alternative='two-sided').pvalue

numeric_comparison = pd.DataFrame(
    {
        'feature': ['age', 'balance', 'campaign'],
        'median_subscribed': [
            subscribed['age'].median(),
            subscribed['balance'].median(),
            subscribed['campaign'].median(),
        ],
        'median_not_subscribed': [
            not_subscribed['age'].median(),
            not_subscribed['balance'].median(),
            not_subscribed['campaign'].median(),
        ],
        'p_value': [age_p, balance_p, campaign_p],
    }
)

display(numeric_comparison)


,feature,median_subscribed,median_not_subscribed,p_value
0,age,38.0,39.0,6.281791e-02
1,balance,733.0,417.0,6.593846e-101
2,campaign,2.0,2.0,1.948490e-71
